# Survivorship Bias: Missing Failures Are Still Evidence

## Missing rows and missing values are clues, not noise

Run the cell below first. It enlarges the font for both code and markdown so the notebook is easy to read while walking through it in class.

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<style>
/* Rendered markdown */
.jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.rendered_html {
    font-size: 24px !important;
    line-height: 1.5 !important;
}
.jp-RenderedHTMLCommon h1, .rendered_html h1 { font-size: 40px !important; }
.jp-RenderedHTMLCommon h2, .rendered_html h2 { font-size: 34px !important; }
.jp-RenderedHTMLCommon h3, .rendered_html h3 { font-size: 30px !important; }
.jp-RenderedHTMLCommon h4, .rendered_html h4 { font-size: 28px !important; }
.jp-RenderedHTMLCommon table, .rendered_html table {
    font-size: 22px !important;
}

/* Code editor (CodeMirror, used by classic + JupyterLab) */
.CodeMirror, .cm-editor, .jp-Editor, .jp-InputArea-editor {
    font-size: 24px !important;
}
.cm-content, .cm-line { font-size: 24px !important; }

/* Code output (print, tracebacks, DataFrame text) */
.jp-OutputArea-output,
.output_area,
.output pre,
.jp-RenderedText pre {
    font-size: 22px !important;
}

/* DataFrame tables in output */
.dataframe, .dataframe th, .dataframe td {
    font-size: 22px !important;
}
</style>
"""))


### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

## The story

Revenue exists for surviving startups. Failed startups often have missing revenue, or vanish from the dataset entirely. If you only analyze survivors, you are studying winners and calling it the population.

## 1. Load the startup dataset

In [ ]:
startups = pd.read_csv("../data/startup_survivorship.csv")
startups.head()

In [ ]:
startups.info()

## 2. Missingness map with `df.isna()`

`isna()` creates a True/False map of missingness.

In [ ]:
startups.isna().head()

## 3. Missing counts with `df.isna().sum()`

Count absence by column.

In [ ]:
startups.isna().sum().sort_values(ascending=False)

## 4. Outcome counts

Survivorship analysis starts with the outcome distribution.

In [ ]:
startups["survived_3yr"].value_counts()

## 5. Cross missingness with the outcome

Is revenue missing because the startup failed?

In [ ]:
pd.crosstab(startups["survived_3yr"], startups["year3_revenue_millions"].isna(),
            rownames=["survived_3yr"], colnames=["revenue_missing"])

## 6. Fill carefully with `df.fillna()`

Filling can be right or wrong depending on meaning. Filling missing revenue with 0 implies the startup earned nothing — but maybe revenue was simply unrecorded.

In [ ]:
startups["year3_revenue_millions"].fillna(0).describe()

## 7. Drop carefully with `df.dropna()`

Dropping missing revenue creates a **survivor-only** table.

In [ ]:
survivors_only = startups.dropna(subset=["year3_revenue_millions"])

## 8. Compare before and after

Always compare shapes and group counts after row removal.

In [ ]:
print("Full dataset shape:   ", startups.shape)
print("Survivors-only shape: ", survivors_only.shape)

print("\nSector counts (full):")
print(startups["sector"].value_counts())

print("\nSector counts (survivors only):")
print(survivors_only["sector"].value_counts())

## 9. The biased headline vs. the honest one

What does the headline 'average startup revenue' look like from each table?

In [ ]:
biased = survivors_only["year3_revenue_millions"].mean()
honest_zero_fill = startups["year3_revenue_millions"].fillna(0).mean()
print(f"Survivor-only average revenue: {biased:.2f}M")
print(f"All startups (failed = 0):     {honest_zero_fill:.2f}M")

## 10. Duplicate rows with `df.duplicated()`

Duplicates are another row-level distortion.

In [ ]:
startups.duplicated().sum()

## 11. Remove duplicates with `df.drop_duplicates()`

Remove duplicates only after checking what they represent.

In [ ]:
clean = startups.drop_duplicates()
print(startups.shape, clean.shape)

## Mini-lab: missing failures

In [ ]:
print(startups.isna().sum())
print(startups["survived_3yr"].value_counts())
survivors_only = startups.dropna(subset=["year3_revenue_millions"])
print(startups.shape, survivors_only.shape)

## Discussion

- What would the dataset look like if it were scraped from success-story blog posts?
- If we report the average revenue from `survivors_only`, what claim are we implicitly making about the failed startups?


## Don't change data silently

Prefer creating a new object over overwriting the original during EDA — your future self will thank you.

In [ ]:
survivors_only = startups.dropna(subset=["year3_revenue_millions"])

## Takeaway

Functions introduced: `isna`, `isna().sum`, `fillna`, `dropna`, `duplicated`, `drop_duplicates`.

**Concept learned: missing rows and missing values are evidence.**